# Python编程基础 · 工单Z《综合项目数据处理系统》
**第12周 | 2课时(80分钟) | 理实一体**  | 教师参考版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 口头架构讲解限时2分钟脱稿——图在心中；示范讲解 |
| **B 组长** | 标准达成 | 终验全过+演示材料课前提交 |
| **C 操作员** | 基础完成 | 可演示版本+checklist过半 |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）

## 一、课前资讯（Predict·预测）
> **预习快问**：checklist三清单(功能/规范/演示)你组还差哪几项？

> **课前检查**：预习微课已观看；跟打练习截图已交；PyCharm CE环境正常

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


工作目录: /Users/joelgong/PycharmProjects/ai-course-teaching-resources/02-程序设计基础


## 三、任务要求（Make）
**实施步骤**
1. 分段验收: 采集→处理→报告模块
2. checklist自查互查
3. 组内预答辩互问热身

**验收标准**：及格(D)=`可演示版本+checklist过半` ；优秀(B+)=`终验全过+演示材料课前提交`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [2]:
# -*- coding: utf-8 -*-
"""W12 综合项目《工业互联网完整数据处理系统》—— 项目模板(骨架)
分工建议(ABCD): 架构/开发/测试/文档, 每人至少负责一个类并答辩一题。
"""
import json
import logging
import random
import time

# 【参考答案】TODO-0 logging配置(复用W06: 控制台+文件, UTF-8; 与w12()一致)
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(levelname)s %(message)s",
                    handlers=[logging.StreamHandler(),
                              logging.FileHandler("w12.log", encoding="utf-8")])
logger = logging.getLogger("system")

### 💡 分层提示（先自己试5分钟再看）
**思路提示**
> - TODO-0: logging配置(复用W06: 控制台+文件, UTF-8)
> - TODO-2(进阶): modbus模式调用W11的read_plc

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位TODO行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改参数/换实现对比效果，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方"验收标准"，及格(D)=`可演示版本+checklist过半`，优秀(B+)=`终验全过+演示材料课前提交`

### Starter 2：`class Collector:`
（Investigate: 读代码→Modify: 补全TODO）

In [3]:
class Collector:
    """采集模块: 优先Modbus(W11); 无环境时用simulate()离线模式"""

    def __init__(self, mode="sim"):
        self.mode = mode

    def simulate(self, n=60):
        """TODO-1 完成态：生成 n 条模拟读数，约 10% 置为空值/err 作为脏数据样本"""
        rows = []
        for i in range(n):
            t = round(random.uniform(45, 100), 1)
            if random.random() < 0.1:
                t = random.choice(["", "err"])
            rows.append({"device_id": f"DEV-{i % 3 + 1:02d}",
                         "ts": time.strftime("%H:%M:%S"),
                         "temp": t, "press": round(random.uniform(.3, .6), 2)})
        return rows

    def read_plc(self):
        """复用W11读数逻辑: 读5个寄存器÷10还原工程值(status除外);
        无pymodbus/模拟器环境时抛异常, 由collect()降级(与w12()降级路径一致)"""
        try:
            from pymodbus.client import ModbusTcpClient as Client
        except ImportError as exc:
            raise RuntimeError(f"pymodbus不可用({exc})") from exc
        c = Client(host="127.0.0.1", port=5020)
        if not c.connect():
            raise RuntimeError("PLC连接失败(127.0.0.1:5020)")
        try:
            rr = c.read_holding_registers(address=0, count=5)
            regs = rr.registers
            return [{"device_id": "DEV-01", "ts": time.strftime("%H:%M:%S"),
                     "temp": regs[0] / 10, "press": regs[1] / 10,
                     "vib": regs[2] / 10, "rpm": regs[3] / 10,
                     "status": regs[4]}]
        finally:
            c.close()

    def collect(self):
        # 【参考答案】TODO-2(进阶) modbus模式调用read_plc; 失败降级simulate(与w12()一致)
        rows = []
        if self.mode == "modbus":
            try:
                rows = self.read_plc()
            except Exception as exc:
                print("  [降级] modbus采集不可用:", exc)
        if not rows:
            rows = self.simulate()
        return rows

### Starter 3：`class Cleaner:`
（Investigate: 读代码→Modify: 补全TODO）

In [4]:
class Cleaner:
    """TODO-3 完成态：清洗模块复用 W08 三维验证（缺失/类型/范围）"""

    def validate(self, row):
        """三维验证: 缺失/类型/范围(与w08()validate完全一致)"""
        t = row["temp"]
        if t in ("", None):
            return "缺失"
        try:
            t = float(t)
        except ValueError:
            return "类型"
        if not 0 <= t <= 120:
            return "范围"
        return None

    def clean(self, rows):
        # 【参考答案】TODO-3 返回(干净行, 问题行), 问题行写bad_records.csv留档(与w12()一致)
        import csv
        ok, bad = [], []
        counter = {"缺失": 0, "类型": 0, "范围": 0}
        for row in rows:
            err = self.validate(row)
            if err is None:
                row["temp"] = float(row["temp"])          # 类型归一
                ok.append(row)
            else:
                bad.append({"device_id": row["device_id"],
                            "temp": row["temp"], "err": err})
                counter[err] += 1
        with open("bad_records.csv", "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)                            # 坏案例留档(同W08)
            w.writerow(["device_id", "temp", "err"])
            for r in bad:
                w.writerow([r["device_id"], r["temp"], r["err"]])
        self.counter, self.bad_rows = counter, bad
        return ok, bad

### Starter 4：`class Analyzer:`
（Investigate: 读代码→Modify: 补全TODO）

In [5]:
class Analyzer:
    """TODO-4 完成态：阈值异常检测+均值/最大/最小统计(与w12()一致)"""

    def __init__(self, warn=80.0, danger=95.0):
        """W02阈值口径: 预警80℃/危险95℃"""
        self.warn, self.danger = warn, danger

    def level(self, temp):
        """阈值分级(与w02()/check同口径): ≥危险→3, ≥预警→2, 其他→0"""
        return 3 if temp >= self.danger else (2 if temp >= self.warn else 0)

    def analyze(self, rows):
        # 【参考答案】TODO-4 阈值异常检测+均值/最大/最小统计(与w12()一致)
        temps = [float(r["temp"]) for r in rows]
        alerts = [{"device_id": r["device_id"], "temp": r["temp"],
                   "level": self.level(r["temp"])}
                  for r in rows if self.level(r["temp"])]
        n3 = sum(1 for a in alerts if a["level"] == 3)
        return {"alerts": alerts,
                "mean_temp": sum(temps) / len(temps),
                "max_temp": max(temps), "min_temp": min(temps), "n": len(temps),
                "n_clean": len(temps), "n_alerts": len(alerts),
                "n_warn": len(alerts) - n3, "n_danger": n3,
                "warn": self.warn, "danger": self.danger,
                "rows": rows}

### Starter 5：`class Reporter:`
（Investigate: 读代码→Modify: 补全TODO）

In [6]:
class Reporter:
    """TODO-5 完成态：reportlab生成PDF: 标题+结论+PIL标注图+GRID统计表(与w12()一致)"""

    def build(self, stats, path="项目报告.pdf"):
        # 【参考答案】TODO-5 reportlab生成PDF: 标题+结论+PIL标注图+GRID统计表(与w12()一致)
        import os

        from reportlab.lib.colors import HexColor
        from reportlab.lib.pagesizes import A4
        from reportlab.lib.styles import getSampleStyleSheet
        from reportlab.lib.units import mm
        from reportlab.pdfbase import pdfmetrics
        from reportlab.pdfbase.ttfonts import TTFont
        from reportlab.platypus import (Image as RLImage, Paragraph,
                                        SimpleDocTemplate, Table, TableStyle)
        # 注: W09词云(优秀)executed版(w12())未覆盖, 需要时可复用W09 main()生成cloud.png
        font = next((f"fonts/{n}" for n in ("ZH_Font.ttf", "SimHei.ttf")
                     if os.path.exists(f"fonts/{n}")),
                    next((p for p in ("/System/Library/Fonts/Supplemental/"
                                      "Arial Unicode MS.ttf",
                                      "/System/Library/Fonts/PingFang.ttc",
                                      "/System/Library/Fonts/STHeiti Light.ttc")
                          if os.path.exists(p)), "SimHei.ttf"))
        pdfmetrics.registerFont(TTFont("SimHei", font))
        img = self.annotate(stats)
        lv3 = sum(1 for a in stats["alerts"] if a["level"] == 3)
        n_raw = stats.get("n_raw", stats.get("n", 0))
        st = getSampleStyleSheet()
        st["Title"].fontName = st["Normal"].fontName = "SimHei"
        tbl_rows = [["指标", "数值"],
                    ["干净记录", f"{stats['n_clean']} 条"],
                    ["告警记录", f"{stats['n_alerts']} 条"
                                f"(警告{stats['n_alerts'] - lv3}/停机{lv3})"],
                    ["平均温度", f"{stats['mean_temp']:.1f}℃"],
                    ["最高温度", f"{stats['max_temp']:.1f}℃"],
                    ["最低温度", f"{stats['min_temp']:.1f}℃"]]
        if "n_raw" in stats and "n_bad" in stats:
            bad = stats["bad_counter"]
            tbl_rows.insert(1, ["原始记录", f"{stats['n_raw']} 条"])
            tbl_rows.insert(2, ["问题记录", f"{stats['n_bad']} 条"
                                          f"(缺失{bad['缺失']}"
                                          f"/类型{bad['类型']}"
                                          f"/范围{bad['范围']})"])
        tbl = Table(tbl_rows)
        tbl.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), 0.5, "black"),
                                 ("FONTNAME", (0, 0), (-1, -1), "SimHei"),
                                 ("BACKGROUND", (0, 0), (-1, 0),
                                  HexColor(0xE8EEF7))]))
        SimpleDocTemplate(path, pagesize=A4).build([
            Paragraph("综合项目日报 · 工业互联网数据处理系统", st["Title"]),
            Paragraph(f"生成时间 {time.strftime('%Y-%m-%d %H:%M:%S')}",
                      st["Normal"]),
            Paragraph(f"结论: 本批次采集{n_raw}条, 干净{stats['n_clean']}条; "
                      f"阈值检测(预警{stats['warn']:.0f}℃/危险{stats['danger']:.0f}℃)"
                      f"触发告警{stats['n_alerts']}条, 其中停机{lv3}条; "
                      f"批次平均温度{stats['mean_temp']:.1f}℃, "
                      f"最高{stats['max_temp']:.1f}℃。"
                      f"建议对超温设备复检并核查液压与冷却系统。", st["Normal"]),
            RLImage(img, width=150 * mm, height=63 * mm),
            tbl])
        print("已生成", path)
        return path

    def annotate(self, stats, path="project_marked.png"):
        # 【参考答案】TODO-5 PIL标注图: 温度折线+预警/危险阈值线+告警红圈(与w12()annotate一致)
        import os

        from PIL import Image, ImageDraw, ImageFont
        W, H = 900, 380
        img = Image.new("RGB", (W, H), "white")
        d = ImageDraw.Draw(img)
        font = next((f"fonts/{n}" for n in ("ZH_Font.ttf", "SimHei.ttf")
                     if os.path.exists(f"fonts/{n}")), "SimHei.ttf")
        try:
            fnt = ImageFont.truetype(font, 16)
            big = ImageFont.truetype(font, 20)
        except Exception:
            fnt = big = ImageFont.load_default()
        rows, alerts = stats.get("rows", []), stats["alerts"]
        warn, danger = stats["warn"], stats["danger"]
        temps = [float(r["temp"]) for r in rows]
        lo, hi = 40, 100
        L, R, T, B = 60, W - 30, 60, H - 40

        def px(i):
            return L + (R - L) * i / max(len(temps) - 1, 1)

        def py(t):
            return B - (B - T) * (t - lo) / (hi - lo)

        for g in range(50, 101, 10):
            d.line([L, py(g), R, py(g)], fill=(228, 228, 228))
            d.text((8, py(g) - 8), f"{g}℃", fill=(90, 90, 90), font=fnt)
        d.line([L, py(warn), R, py(warn)], fill=(230, 145, 20))
        d.text((R - 160, py(warn) - 24), f"预警线 {warn:.0f}℃",
               fill=(190, 110, 10), font=fnt)
        d.line([L, py(danger), R, py(danger)], fill=(200, 30, 30))
        d.text((R - 160, py(danger) - 24), f"危险线 {danger:.0f}℃",
               fill=(200, 30, 30), font=fnt)
        d.line([L, B, R, B], fill=(0, 0, 0), width=2)
        pts = [(px(i), py(t)) for i, t in enumerate(temps)]
        d.line(pts, fill=(43, 108, 176), width=2)
        hot = {(a["device_id"], a["temp"]) for a in alerts}
        for i, (x, y) in enumerate(pts):
            if (rows[i]["device_id"], rows[i]["temp"]) in hot:
                d.ellipse([x - 5, y - 5, x + 5, y + 5],
                          outline=(200, 30, 30), width=3)
        d.text((L, 14), "批次温度曲线与阈值标注(PIL)", fill=(20, 20, 20),
               font=big)
        img.save(path)
        return path

### Starter 6：`def main():`
（Investigate: 读代码→Modify: 补全TODO）

In [7]:
def main():
    rows = Collector().collect()
    clean, bad = Cleaner().clean(rows)
    stats = Analyzer().analyze(clean)
    print("原始%d 干净%d 问题%d" % (len(rows), len(clean), len(bad)))
    print(stats)
    Reporter().build(stats)

### Starter 7：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [8]:
if __name__ == "__main__":
    main()

原始60 干净54 问题6
{'alerts': [{'device_id': 'DEV-03', 'temp': 92.3, 'level': 2}, {'device_id': 'DEV-02', 'temp': 96.2, 'level': 3}, {'device_id': 'DEV-02', 'temp': 85.5, 'level': 2}, {'device_id': 'DEV-03', 'temp': 98.9, 'level': 3}, {'device_id': 'DEV-01', 'temp': 83.4, 'level': 2}, {'device_id': 'DEV-03', 'temp': 81.1, 'level': 2}, {'device_id': 'DEV-01', 'temp': 83.5, 'level': 2}, {'device_id': 'DEV-01', 'temp': 86.8, 'level': 2}, {'device_id': 'DEV-03', 'temp': 93.1, 'level': 2}, {'device_id': 'DEV-03', 'temp': 84.1, 'level': 2}, {'device_id': 'DEV-03', 'temp': 88.4, 'level': 2}, {'device_id': 'DEV-01', 'temp': 82.8, 'level': 2}, {'device_id': 'DEV-01', 'temp': 86.9, 'level': 2}, {'device_id': 'DEV-01', 'temp': 95.3, 'level': 3}, {'device_id': 'DEV-01', 'temp': 84.1, 'level': 2}, {'device_id': 'DEV-02', 'temp': 81.9, 'level': 2}, {'device_id': 'DEV-02', 'temp': 81.9, 'level': 2}], 'mean_temp': 70.79814814814814, 'max_temp': 98.9, 'min_temp': 46.3, 'n': 54, 'n_clean': 54, 'n_alerts': 17

已生成 项目报告.pdf


## 四、提交要求与避坑
- 提交: `学号_姓名_Z.py` 或本notebook(含输出)
- ⚠ 演示视频课前提交（≤3分钟），教师预看带问题入场
- ⚠ >6组切换5+3分钟制，超时扣表达分
- ⚠ 脏数据bad-case演示必准备（评委必点）

---

## 教师参考答案（来源：`教师参考/solutions.py` · `w12()`）

> **发放纪律：本附录勿随学生版下发。** 课堂按层放提示（D层演示照做/C层只给思路行/A层直接做）。

学生版 TODO 位置 → 下方参考实现的对应关系：

| 学生版 TODO 函数 | 参考实现说明 |
|---|---|

In [9]:
def w12():
    """综合项目: 直接组合w08清洗+w02阈值+w10报告"""
    print("=== 综合系统演示 ===")
    w08()          # 采集(模拟)+清洗
    w02()          # 异常检测(阈值)
    print("报告生成复用 w10() ...")

## 运行与验证

上方参考实现可**直接运行**（合成数据、CPU 离线、结果可复现）。
教师可对照学生版逐 cell 讲解：学生填 TODO → 此处给出完成态。

> 多解法对比见同目录《多解法手册.md》与 `教师参考/solutions_multi*.py`。